In [ ]:
# mount google drive (used google drive to make use of T4 GPU)
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
# === Imports ===
import pandas as pd
import torch
import torch.nn as nn
import joblib
from transformers import RobertaTokenizer, RobertaModel
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm

In [ ]:
# === Define Model ===
class RobertaBiLSTMModel(nn.Module):
    def __init__(self, hidden_dim=128, output_dim=2):
        super(RobertaBiLSTMModel, self).__init__()
        self.roberta = RobertaModel.from_pretrained("roberta-base")
        self.lstm = nn.LSTM(
            self.roberta.config.hidden_size,
            hidden_dim,
            batch_first=True,
            bidirectional=True
        )
        self.fc = nn.Linear(hidden_dim * 2, output_dim)
        self.dropout = nn.Dropout(0.3)

    def forward(self, input_ids, attention_mask):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        lstm_out, _ = self.lstm(outputs.last_hidden_state)
        pooled_output = self.dropout(lstm_out[:, -1, :])
        return self.fc(pooled_output)


# Sentiment Analysis

Steps to label the data:
1. **Load the data**: Load the dataset containing the text data and their corresponding labels.
2. **Perform Subjectivity Detection**: Use a pre-trained subjectivity detection model to classify the text data into subjective and objective (Neutral) categories.
3. **Perform Sarcasm Detection**: Use a pre-trained sarcasm detection model to classify the remaining text data into sarcastic (Negative) and non-sarcastic categories.
4. **Perform Sentiment Analysis**: Use a pre-trained sentiment analysis model to classify the remaining text data into positive, negative categories.

In [ ]:
# === Load Data ===
file_path = "/content/drive/MyDrive/SC4021/Data/consolidated_reviews/combined_data.csv"
data = pd.read_csv(file_path, lineterminator="\n")
print(f"Loaded {len(data)} rows from {file_path}")

# Initialize 'sentiment' column
data["sentiment"] = None

# === Device Setup ===
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

<ipython-input-10-dd1742c93de7>:3: DtypeWarning: Columns (0,1,2,8,9,11,12) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv(file_path, lineterminator="\n")


Loaded 1248415 rows from /content/drive/MyDrive/SC4021/Data/consolidated_reviews/combined_data.csv


## Subjectivity Detection


In [ ]:
# === Subjectivity Detection ===
print("Starting Subjectivity Detection...")

# 1. TF-IDF Vectorization
vectorizer = joblib.load("/content/drive/MyDrive/SC4021/Best Combination Weights/best_subjectivity_vectorizer.pkl")
X_tfidf = vectorizer.transform(data["review_text"])
print("TF-IDF Vectorizer Shape:", X_tfidf.shape)

# 2. Load Subjectivity Model and Predict
subjectivity_model = joblib.load("/content/drive/MyDrive/SC4021/Best Combination Weights/best_subjectivity_detection_model.pkl")
subjectivity_labels = subjectivity_model.predict(X_tfidf)

# 3. Update DataFrame
data["subjectivity"] = subjectivity_labels
print("Subjectivity detection completed.")

data.loc[data["subjectivity"] == "Objective", "sentiment"] = "neutral"

Starting Subjectivity Detection...
TF-IDF Vectorizer Shape: (1248415, 8982)
Subjectivity detection completed.


In [ ]:
print(data["sentiment"].value_counts())
print(data.info())

sentiment
neutral    8135
Name: count, dtype: int64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1248415 entries, 0 to 1248414
Data columns (total 20 columns):
 #   Column                  Non-Null Count    Dtype  
---  ------                  --------------    -----  
 0   gaming_platform         25890 non-null    object 
 1   review_type             25890 non-null    object 
 2   username                25890 non-null    object 
 3   score                   25834 non-null    float64
 4   timestamp_updated_date  1248401 non-null  object 
 5   review_text             1248415 non-null  object 
 6   platform                1248415 non-null  object 
 7   game                    1248415 non-null  object 
 8   Title                   297570 non-null   object 
 9   URL                     297570 non-null   object 
 10  Score                   297570 non-null   float64
 11  Subreddit               297570 non-null   object 
 12  Comment Author          297168 non-null   object 
 13  Level

## Sarcasm Detection

In [ ]:
# === Sarcasm Detection ===
print("Starting Sarcasm Detection...")

# 1. Load Model
sarcasm_model = RobertaBiLSTMModel().to(device)
sarcasm_model.load_state_dict(torch.load("/content/drive/MyDrive/SC4021/Best Combination Weights/roberta_bilstm_sarcasm.pth", map_location=device))
sarcasm_model.eval()

# 2. Prepare Data
subjective_texts = data[data["subjectivity"] == "Subjective"]["review_text"].tolist()

encodings = tokenizer(subjective_texts, truncation=True, padding=True, max_length=128, return_tensors="pt")

input_ids = encodings["input_ids"]
attention_mask = encodings["attention_mask"]

dataset = TensorDataset(input_ids, attention_mask)
loader = DataLoader(dataset, batch_size=128)  # Faster

# 3. Prediction Loop with tqdm
preds = []
label_maps = {0: "not sarcasm", 1: "sarcasm"}




Starting Sarcasm Detection...


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
with torch.no_grad():
    for batch in tqdm(loader, desc="Predicting sarcasm", unit="batch"):
        batch_input_ids, batch_attention_mask = [b.to(device) for b in batch]
        outputs = sarcasm_model(batch_input_ids, batch_attention_mask)
        batch_preds = torch.argmax(outputs, dim=1)
        preds.extend([label_maps[p.item()] for p in batch_preds])

print("Sarcasm detection completed.")

# 4. Update DataFrame
data.loc[data["subjectivity"] == "Subjective", "sarcasm"] = preds

# 5. Set Sentiment Negative for Sarcasm
data.loc[data["sarcasm"] == "sarcasm", "sentiment"] = "negative"

Predicting sarcasm: 100%|██████████| 9690/9690 [2:09:23<00:00,  1.25batch/s]


Sarcasm detection completed.


In [ ]:
print(data["sentiment"].value_counts())

sentiment
negative    217494
neutral       8135
Name: count, dtype: int64


In [ ]:
output_file_path = "/content/drive/MyDrive/SC4021/Data/tmp_combined_data.csv"
data.to_csv(output_file_path, index=False)

## Polarity Detection

In [ ]:
# === Imports ===
import pandas as pd
import torch
import torch.nn as nn
import joblib
from transformers import RobertaTokenizer, RobertaModel
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm

In [ ]:
# === Define Model ===
class RobertaBiLSTMModel(nn.Module):
    def __init__(self, hidden_dim=128, output_dim=2):
        super(RobertaBiLSTMModel, self).__init__()
        self.roberta = RobertaModel.from_pretrained("roberta-base")
        self.lstm = nn.LSTM(
            self.roberta.config.hidden_size,
            hidden_dim,
            batch_first=True,
            bidirectional=True
        )
        self.fc = nn.Linear(hidden_dim * 2, output_dim)
        self.dropout = nn.Dropout(0.3)

    def forward(self, input_ids, attention_mask):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        lstm_out, _ = self.lstm(outputs.last_hidden_state)
        pooled_output = self.dropout(lstm_out[:, -1, :])
        return self.fc(pooled_output)


In [ ]:
# === Device Setup ===
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

In [ ]:
# === Load Data ===
file_path = "/content/drive/MyDrive/SC4021/Data/tmp_combined_data.csv"
data = pd.read_csv(file_path, lineterminator="\n")
print(f"Loaded {len(data)} rows from {file_path}")

<ipython-input-5-d7ee4e59f7c5>:3: DtypeWarning: Columns (0,1,2,8,9,11,12) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv(file_path, lineterminator="\n")


Loaded 1248415 rows from /content/drive/MyDrive/SC4021/Data/tmp_combined_data.csv


In [ ]:
# === Polarity Detection ===
print("Starting Polarity Detection...")
# 1. Load Polarity Model
polarity_model = RobertaBiLSTMModel().to(device)
polarity_model.load_state_dict(torch.load("/content/drive/MyDrive/SC4021/Best Combination Weights/roberta_bilstm_polarity.pth", map_location=device))
polarity_model.eval()

# 2. Prepare Data
non_sarcasm_texts = data[data["sentiment"].isna()]["review_text"].tolist()

encodings = tokenizer(non_sarcasm_texts, truncation=True, padding=True, max_length=128, return_tensors="pt")

input_ids = encodings["input_ids"]
attention_mask = encodings["attention_mask"]

dataset = TensorDataset(input_ids, attention_mask)
loader = DataLoader(dataset, batch_size=256)  # Faster

# 3. Prediction Loop with tqdm
preds = []
label_maps = {0: "negative", 1: "positive"}


Starting Polarity Detection...


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
with torch.no_grad():
    for batch in tqdm(loader, desc="Predicting sarcasm", unit="batch"):
        batch_input_ids, batch_attention_mask = [b.to(device) for b in batch]
        outputs = polarity_model(batch_input_ids, batch_attention_mask)
        batch_preds = torch.argmax(outputs, dim=1)
        preds.extend([label_maps[p.item()] for p in batch_preds])

print("Polarity detection completed.")

Predicting sarcasm: 100%|██████████| 3996/3996 [1:52:41<00:00,  1.69s/batch]

Polarity detection completed.


NameError: name 'polarity_preds' is not defined

In [ ]:
# 4. Update DataFrame
data.loc[(data["sarcasm"] == "not sarcasm") & (data["sentiment"].isna()), "sentiment"] = preds

In [ ]:
print(data["sentiment"].value_counts())
print(data.info())

sentiment
positive    632937
negative    607343
neutral       8135
Name: count, dtype: int64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1248415 entries, 0 to 1248414
Data columns (total 21 columns):
 #   Column                  Non-Null Count    Dtype  
---  ------                  --------------    -----  
 0   gaming_platform         25890 non-null    object 
 1   review_type             25890 non-null    object 
 2   username                25890 non-null    object 
 3   score                   25834 non-null    float64
 4   timestamp_updated_date  1248401 non-null  object 
 5   review_text             1248415 non-null  object 
 6   platform                1248415 non-null  object 
 7   game                    1248415 non-null  object 
 8   Title                   297570 non-null   object 
 9   URL                     297570 non-null   object 
 10  Score                   297570 non-null   float64
 11  Subreddit               297570 non-null   object 
 12  Comment Author     

## Save the Data

In [ ]:
# === Save Final Output ===
output_file_path = "/content/drive/MyDrive/SC4021/Data/labelled_combined_data.csv"
data.to_csv(output_file_path, index=False)
print(f"Final data saved to {output_file_path}")
